In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv
import numpy as np

from problems.vrp.problem_class import CeohRoutefinderProblem

from rl4co.data.utils import load_npz_to_tensordict

from libraries.routefinder.baselines.pyvrp import instance2data
from libraries.routefinder.baselines.ortools import instance2data as instance2data_or_tools
from libraries.routefinder.baselines.constants import PYVRP_SCALING_FACTOR, ORTOOLS_SCALING_FACTOR
from libraries.routefinder.envs.mtdvrp import MTVRPEnv
from libraries.routefinder.baselines.utils import mtvrp2anyvrp

In [2]:
# Configuration
# load BASE_PATH from .env file
load_dotenv()
BASE_PATH = os.getenv("BASE_PATH")

#problem_name ="ovrpmbltw"
problem_name = "vrptw"

instance_class = "test"
instance_file = "50"

instance_index = 1

data_files = []
for root, dirs, files in os.walk("../../../data/vrp_instances"):
     for file in files:
        if file == "50.npz" or file == "100.npz":
            data_files.append(os.path.join(root, file))

data_files = sorted(data_files, key=lambda x: len(x))
data_files = sorted(data_files, key=lambda x: x.split("/")[-2])
data_files = sorted(data_files, key=lambda x: int(x.split("/")[-1].split(".")[0]))


file = ""
for f in data_files:
    if instance_class in f and instance_file in f:
        file = f

if file == "":
    exit(0)


In [3]:
env = MTVRPEnv(check_solution=False)

td_test = load_npz_to_tensordict(file)
td_test = env.reset(td_test)

instances = mtvrp2anyvrp(td_test)

pyvrp_data = instance2data(instances[instance_index], ORTOOLS_SCALING_FACTOR)

problem = CeohRoutefinderProblem(problem_name)

eoh_data = problem.get_datasets_for_file(instance_class, instance_file)[instance_index]


/home/nico/anaconda3/envs/routefinder/lib/python3.10/site-packages/torchrl/data/tensor_specs.py:6294: DeprecationWarning: The BoundedTensorSpec has been deprecated and will be removed in v0.8. Please use Bounded instead.
  warnings.warn(
/home/nico/anaconda3/envs/routefinder/lib/python3.10/site-packages/torchrl/data/tensor_specs.py:6294: DeprecationWarning: The UnboundedDiscreteTensorSpec has been deprecated and will be removed in v0.8. Please use Unbounded instead.
  warnings.warn(
/home/nico/anaconda3/envs/routefinder/lib/python3.10/site-packages/torchrl/data/tensor_specs.py:6294: DeprecationWarning: The CompositeSpec has been deprecated and will be removed in v0.8. Please use Composite instead.
  warnings.warn(
/home/nico/anaconda3/envs/routefinder/lib/python3.10/site-packages/torchrl/data/tensor_specs.py:6294: DeprecationWarning: The UnboundedContinuousTensorSpec has been deprecated and will be removed in v0.8. Please use Unbounded instead.
  warnings.warn(
/home/nico/.local/lib/py

[WORKER] PID=735748 started worker_main!


In [4]:
ortools_data = instance2data_or_tools(instances[instance_index])

In [5]:
from external_libraries.routefinder.util.data_creation import create_dataset, build_routing_model

datasets = create_dataset(file)
select = datasets[instance_index]

m,t = build_routing_model(select)


In [6]:
eoh_data = select

# Experiment Data

In [7]:
ceoh_experimen_data_equals_routefinder = (
    eoh_data['max_distance'] == ortools_data.max_distance and
    eoh_data['num_vehicles'] == ortools_data.num_vehicles and
    len(eoh_data['depots']) == ortools_data.num_depots
)

if ceoh_experimen_data_equals_routefinder:
    print("Experiment data is EQUAL!")
else:
    print("[ERROR] Experiment data NOT EQUAL!")

Experiment data is EQUAL!


# Time Windows

In [8]:

rf_early = ortools_data.vehicle_tw_early
eoh_early = eoh_data['vehicle_tw_early']

rf_late = ortools_data.vehicle_tw_late
eoh_late = eoh_data['vehicle_tw_late']

rf_tw = ortools_data.time_windows
eoh_tw = eoh_data['time_windows']


ceoh_data_equals_routefinder = (
    (np.array(rf_early) == np.array(eoh_early)).all() and
    (np.array(rf_late) == np.array(eoh_late)).all() and
    (np.array(rf_tw) == np.array(eoh_tw)).all()
)

if ceoh_data_equals_routefinder:
    print("Time Windows are EQUAL!")
else:
    print("[ERROR] Time Windows NOT EQUAL!")


Time Windows are EQUAL!


# Experiment Configs

In [9]:
ceoh_exp_config_equals_routefinder = (
    (np.array(eoh_data['distance_matrix']) == np.array(ortools_data.distance_matrix)).all() and
    (np.array(eoh_data['duration_matrix']) == np.array(ortools_data.duration_matrix)).all() and
    (np.array(eoh_data['demands']) == np.array(ortools_data.demands)).all() and
    (np.array(eoh_data['backhauls']) == np.array(ortools_data.backhauls)).all()
)

if ceoh_exp_config_equals_routefinder:
    print("Experiment configs are EQUAL!")
else:
    print("[ERROR] Experiment configs NOT EQUAL!")

Experiment configs are EQUAL!


# Problem Data

In [10]:
t1 = np.array(eoh_data['duration_matrix'])
t2 = np.array(ortools_data.duration_matrix)


def transform_matrix(A):
    n = A.shape[0]
    diag = np.diag(A)
    B = A.copy()

    # Keep the first row as-is
    B[0] = A[0]

    for i in range(1, n):
        B[i, 0] = diag[i]
        B[i, i] = diag[i]

    return B

t1 = transform_matrix(t1)

print(t1)
print(t2)

(t1 == t2).all()


[[     0  44401  76807 ...  38783  37124  40933]
 [ 15753  15753 119485 ...  97249  89172 101073]
 [ 17905 121637  17905 ... 102803 117097  86530]
 ...
 [ 16769  98265 101667 ...  16769  35570  34303]
 [ 16281  89700 115473 ...  35082  16281  51760]
 [ 17793 103113  86418 ...  35327  53272  17793]]
[[     0  44401  76807 ...  38783  37124  40933]
 [ 15753  15753 119485 ...  97249  89172 101073]
 [ 17905 121637  17905 ... 102803 117097  86530]
 ...
 [ 16769  98265 101667 ...  16769  35570  34303]
 [ 16281  89700 115473 ...  35082  16281  51760]
 [ 17793 103113  86418 ...  35327  53272  17793]]


True

In [11]:
def transform_matrix(A):
    n = A.shape[0]
    diag = np.diag(A)
    B = A.copy()

    # Keep the first row as-is
    B[0] = A[0]

    for i in range(1, n):
        B[i, 0] = diag[i]
        B[i, i] = diag[i]

    return B

transform_matrix(t1)


array([[     0,  44401,  76807, ...,  38783,  37124,  40933],
       [ 15753,  15753, 119485, ...,  97249,  89172, 101073],
       [ 17905, 121637,  17905, ..., 102803, 117097,  86530],
       ...,
       [ 16769,  98265, 101667, ...,  16769,  35570,  34303],
       [ 16281,  89700, 115473, ...,  35082,  16281,  51760],
       [ 17793, 103113,  86418, ...,  35327,  53272,  17793]])

In [12]:
t1 = np.array(eoh_data['vehicle_start_depots'])
t2 = np.array(ortools_data.vehicle_start_depots)
t2 = np.array(ortools_data.depots)


In [13]:
print((np.array(ortools_data.distance_matrix) == np.array(ortools_data.duration_matrix)).all())

False
